In [1]:
# Kaggle setup: clone repo, install libraries, find the data,
# copy results.csv (from 05) and the trained YOLO weights (from 04).
import os, glob, shutil, subprocess

REPO = "polluvl/hw2-video-detection"

if os.path.isdir("/kaggle/working"):
    # 1. clone the repo once per session
    if not os.path.isdir("/kaggle/working/repo"):
        r = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", "/kaggle/working/repo"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git clone failed (is Internet on?):\n" + r.stderr)
    # 2. install the pinned libraries
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
    os.chdir("/kaggle/working/repo/notebooks")

    # 3. find where Kaggle mounted the competition data
    hits = glob.glob("/kaggle/input/**/test_images.csv", recursive=True)
    if not hits:
        raise RuntimeError("Competition data not found: add it in the right panel under Input")
    os.environ["DATA_ROOT"] = os.path.dirname(hits[0])
    print("data root:", os.environ["DATA_ROOT"])

    # 4. results.csv from the 05 output (it already contains the 04 YOLO rows)
    found05 = glob.glob("/kaggle/input/**/preds_val_rtdetr_640.npz", recursive=True)
    if not found05:
        raise RuntimeError("05 output not found: Add Input -> Your Work -> the RT-DETR notebook")
    shutil.copy(os.path.join(os.path.dirname(found05[0]), "results.csv"), "/kaggle/working/results.csv")

    # 5. trained YOLO weights from the 04 output
    found04 = glob.glob("/kaggle/input/**/ultralytics/yolo_1024/weights/best.pt", recursive=True)
    if not found04:
        raise RuntimeError("04 output not found: Add Input -> Your Work -> 04_yolo")
    ultra04 = found04[0].split("/yolo_1024/")[0]
    for run_dir in glob.glob(f"{ultra04}/yolo_*"):
        name = os.path.basename(run_dir)
        os.makedirs(f"/kaggle/working/ultralytics/{name}/weights", exist_ok=True)
        shutil.copy(f"{run_dir}/weights/best.pt", f"/kaggle/working/ultralytics/{name}/weights/")
        print("copied weights:", name)

data root: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
copied weights: yolo_1024
copied weights: yolo_1024_box
copied weights: yolo_1024_p2
copied weights: yolo_640


# 06 · Final pipeline
Starts from the best detector of 02–05 (by our val mAP) and changes **one thing at a time**,
keeping two groups apart as the assignment asks:
- **A. data / model changes** (need retraining): more training frames; a bigger model
- **B. inference changes** (same weights): test resolution, flip TTA, tiling, temporal rescoring,
  ensembling with WBF

All B experiments are evaluated on **all** val frames (step 1), because temporal rescoring needs
consecutive frames. These numbers are therefore not directly comparable to the step-5 numbers
of 02–05; the first row (`pipe_base`) is the reference for this table.

In [2]:
import ast
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
from common import (SMOKE, evaluate, evaluate_per_clip, find_data_root,
                    list_frames, list_test_frames, load_preds, log_result,
                    save_preds, seed_everything, work_dir, write_submission)
from inference import merge_wbf, predict_tiled, temporal_rescore
from yolo_utils import export_yolo_split, predict_ultralytics, write_data_yaml

# stop right away if there is no GPU
assert torch.cuda.is_available(), "No GPU! Session options -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

DATA = find_data_root()
OUT = work_dir()
seed_everything(42)
DEVICE = 0 if torch.cuda.is_available() else "cpu"

RUN_DATA_EXP = True    # A1: retrain best config on 3x more frames
RUN_MODEL_EXP = True   # A2: YOLOv8m instead of YOLOv8s

EPOCHS = 1 if SMOKE else 15   # same budget as in 04
results = pd.read_csv(OUT / "results.csv")
yolo_runs = results[results.experiment.str.startswith("yolo")].sort_values("mAP")
BEST = yolo_runs.experiment.iloc[-1]
BEST_SPEC = ast.literal_eval(yolo_runs.config.iloc[-1])  # the dict logged by 04 (imgsz, box, ...)
IMGSZ = BEST_SPEC["imgsz"]
print("starting point:", BEST, BEST_SPEC)
results[["experiment", "family", "mAP", "AP50", "AP_small", "ms_per_img"]].sort_values("mAP")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU: Tesla T4
starting point: yolo_1024_box {'imgsz': 1024, 'box': 10.0}


,experiment,family,mAP,AP50,AP_small,ms_per_img
4,rtdetr_640,query-based,0.0787,0.1510,0.0345,47.0
0,yolo_640,one-stage anchor-free,0.1455,0.2807,0.0731,10.5
6,rtdetr_640_lr,query-based,0.1541,0.2829,0.0600,47.3
5,rtdetr_640_q500,query-based,0.1587,0.2998,0.0612,50.7
2,yolo_1024_p2,one-stage anchor-free,0.1816,0.3657,0.1043,36.6
1,yolo_1024,one-stage anchor-free,0.1821,0.3594,0.1034,23.5
3,yolo_1024_box,one-stage anchor-free,0.1895,0.3625,0.1080,23.7


## A. Data and model changes (evaluated on the same step-5 val frames as 02–05)

In [3]:
val5 = list_frames(DATA / "val", step=1 if SMOKE else 5)
base_model_file = "yolov8s-p2.yaml" if "p2" in BEST else "yolov8s.pt"


def train_yolo(name, model_file, train_step, extra=None, weights=None):
    ds = OUT / f"yolo_ds_step{train_step}"
    if not (ds / "data.yaml").exists():
        export_yolo_split(list_frames(DATA / "train", step=train_step), ds / "train")
        export_yolo_split(val5, ds / "val")
        write_data_yaml(ds / "data.yaml", ds / "train", ds / "val")
    model = YOLO(model_file)
    if weights:
        model.load(weights)
    args = dict(data=str(ds / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=2 if SMOKE else 8,
                workers=2, seed=42, deterministic=True, device=DEVICE, project=str(OUT / "ultralytics"),
                exist_ok=True, cos_lr=True, close_mosaic=5, max_det=500, plots=not SMOKE, name=name)
    args.update({k: v for k, v in BEST_SPEC.items() if k not in ("model", "weights", "imgsz")})
    args.update(extra or {})
    model.train(**args)
    best = YOLO(str(OUT / "ultralytics" / name / "weights" / "best.pt"))
    preds, ms = predict_ultralytics(best, val5, imgsz=IMGSZ)
    save_preds(preds, OUT / f"preds_val_{name}.npz")
    m = evaluate(val5, preds)
    log_result(name, m, {"family": "pipeline (data/model)", "ms_per_img": round(ms, 1),
                         "config": f"{model_file}, train_step={train_step}, {extra or {}}"})
    print(name, m)
    return m


p2_weights = None if SMOKE else ("yolov8s.pt" if "p2" in BEST else None)
if SMOKE:
    base_model_file = base_model_file.replace(".pt", ".yaml")
if RUN_DATA_EXP:  # data change only: every 3rd frame instead of every 10th
    train_yolo("pipe_data_step3", base_model_file, 1 if SMOKE else 3, weights=p2_weights)
if RUN_MODEL_EXP:  # model change only: same data as BEST, bigger network
    m_file = base_model_file.replace("yolov8s", "yolov8m")
    train_yolo("pipe_model_m", m_file, 1 if SMOKE else 10,
               weights=None if SMOKE else ("yolov8m.pt" if "p2" in BEST else None))

results = pd.read_csv(OUT / "results.csv")
cand = results.set_index("experiment").loc[[r for r in [BEST, "pipe_data_step3", "pipe_model_m"]
                                            if r in set(results.experiment)]].reset_index()
FINAL_RUN = cand.sort_values("mAP", ascending=False, kind="stable").experiment.iloc[0]  # ties -> BEST
print(cand[["experiment", "mAP", "AP50", "AP_small", "config"]])
print("model used for the inference experiments:", FINAL_RUN)

New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo_ds_step3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=500, mixup=0.0, mode=train, model=yolov8s.pt,

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/15       7.4G      2.032     0.9018      1.015        145       1024: 100% ━━━━━━━━━━━━ 1012/1012 3.5it/s 4:53
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.5it/s 10.2s
                   all        571      22906      0.564      0.411      0.403      0.214

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/15      6.72G      1.967     0.8437      1.001        162       1024: 100% ━━━━━━━━━━━━ 1012/1012 3.5it/s 4:52
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.6it/s 10.0s
                   all        571      22906      0.495      0.396      0.391      0.197

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/15      6.72G      1.924     0.7959     0.9891        114       1024: 100% ━━━━━━━━━━━━ 1012/1012 3.5it/s 4:53
                 Class     Images  Instances    

## B. Inference strategies (same weights), all val frames

In [4]:
val_all = list_frames(DATA / "val", step=1)
model = YOLO(str(OUT / "ultralytics" / FINAL_RUN / "weights" / "best.pt"))
pipe = {}


def record(name, preds, ms, note):
    save_preds(preds, OUT / f"preds_valall_{name}.npz")
    m = evaluate(val_all, preds)
    log_result(name, m, {"family": "pipeline (inference)", "ms_per_img": round(ms, 1), "config": note})
    pipe[name] = preds
    print(f"{name:22s} mAP {m['mAP']:.4f} AP50 {m['AP50']:.4f} AP_s {m['AP_small']:.4f} ({ms:.0f} ms/img)")
    return m


p, ms = predict_ultralytics(model, val_all, imgsz=IMGSZ)
record("pipe_base", p, ms, f"{FINAL_RUN}, imgsz={IMGSZ}")

big = int(IMGSZ * 1.25 // 32 * 32)
p, ms = predict_ultralytics(model, val_all, imgsz=big)
record("pipe_imgsz_up", p, ms, f"test imgsz={big}")

p, ms = predict_ultralytics(model, val_all, imgsz=IMGSZ, augment=True)
record("pipe_tta", p, ms, "Ultralytics TTA (flip + scales)")

t0 = time.time()
p = predict_tiled(model, val_all, imgsz=IMGSZ, grid=(2, 2), overlap=0.2, merge="nms")
record("pipe_tiles", p, 1000 * (time.time() - t0) / len(val_all), "full image + 2x2 tiles, NMS merge")

pipe_base              mAP 0.2015 AP50 0.3777 AP_s 0.1000 (22 ms/img)
pipe_imgsz_up          mAP 0.2115 AP50 0.3988 AP_s 0.1020 (36 ms/img)
pipe_tta               mAP 0.2019 AP50 0.3874 AP_s 0.1015 (47 ms/img)
pipe_tiles             mAP 0.2230 AP50 0.4291 AP_s 0.1161 (147 ms/img)


{'mAP': 0.223,
 'AP50': 0.4291,
 'AP75': 0.203,
 'AP_small': 0.1161,
 'AP_medium': 0.2038,
 'AP_large': 0.3712,
 'AR': 0.3686}

### Temporal rescoring (bonus) on top of the best single-pass strategy
Small grid over the matching IoU and the smoothing weight, on val only.

In [5]:
pipe_res = pd.read_csv(OUT / "results.csv")
# on a tie keep the earlier = cheaper strategy (stable sort, descending)
pipe_res = pipe_res[pipe_res.experiment.isin(pipe)].sort_values("mAP", ascending=False, kind="stable")
BEST_INF = pipe_res.experiment.iloc[0]
print("best inference strategy so far:", BEST_INF)

grid = []
for iou_thr in [0.2, 0.3, 0.5]:
    for weight in [0.3, 0.5, 0.7]:
        for causal in [False, True]:
            p = temporal_rescore(val_all, pipe[BEST_INF], iou_thr=iou_thr, weight=weight, causal=causal)
            m = evaluate(val_all, p)
            grid.append({"iou_thr": iou_thr, "weight": weight, "causal": causal, **m})
grid = pd.DataFrame(grid).sort_values("mAP", ascending=False)
print(grid[["iou_thr", "weight", "causal", "mAP", "AP50", "AP_small"]].head(8))

best_t = grid.iloc[0]
TEMPORAL = dict(iou_thr=float(best_t.iou_thr), weight=float(best_t.weight), causal=bool(best_t.causal))
p = temporal_rescore(val_all, pipe[BEST_INF], **TEMPORAL)
record("pipe_temporal", p, 0, f"{BEST_INF} + temporal {TEMPORAL}")
print("per-clip mAP before / after temporal rescoring:")
pd.merge(evaluate_per_clip(val_all, pipe[BEST_INF]), evaluate_per_clip(val_all, p),
         on=["clip", "n_frames"], suffixes=("_before", "_after"))

best inference strategy so far: pipe_tiles
    iou_thr  weight  causal     mAP    AP50  AP_small
4       0.2     0.7   False  0.2348  0.4502    0.1201
10      0.3     0.7   False  0.2345  0.4498    0.1202
16      0.5     0.7   False  0.2341  0.4487    0.1209
2       0.2     0.5   False  0.2334  0.4473    0.1208
8       0.3     0.5   False  0.2332  0.4470    0.1206
14      0.5     0.5   False  0.2331  0.4462    0.1208
0       0.2     0.3   False  0.2308  0.4424    0.1195
6       0.3     0.3   False  0.2307  0.4421    0.1193
pipe_temporal          mAP 0.2348 AP50 0.4502 AP_s 0.1201 (0 ms/img)
per-clip mAP before / after temporal rescoring:


,clip,n_frames,mAP_before,AP50_before,mAP_after,AP50_after
0,uav0000086_00000_v,464,0.1714,0.3787,0.1827,0.3962
1,uav0000117_02622_v,349,0.2478,0.5089,0.2570,0.5283
2,uav0000137_00458_v,233,0.2936,0.6226,0.2982,0.6374
3,uav0000182_00000_v,363,0.1778,0.3782,0.1819,0.3828
4,uav0000268_05773_v,978,0.1305,0.2318,0.1360,0.2438
5,uav0000305_00000_v,184,0.2269,0.3284,0.2348,0.3364
6,uav0000339_00001_v,275,0.2641,0.4396,0.2566,0.4325


### Ensemble (WBF) of the two best different models, if both exist
(only meaningful if their errors differ; checked on val before using it)

In [6]:
others = [r for r in [BEST, "pipe_data_step3", "pipe_model_m"]
          if r != FINAL_RUN and (OUT / "ultralytics" / r / "weights" / "best.pt").exists()]
if others:
    second = YOLO(str(OUT / "ultralytics" / others[0] / "weights" / "best.pt"))
    p2, _ = predict_ultralytics(second, val_all, imgsz=IMGSZ)
    ens = {r.image_key: merge_wbf([pipe["pipe_base"][r.image_key], p2[r.image_key]], r.width, r.height)
           for r in val_all.itertuples()}
    record("pipe_wbf_2models", ens, 0, f"WBF({FINAL_RUN}, {others[0]})")

/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [  0.0068101           0     0.16086           0].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.042107           1    0.049547           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.042222           1    0.049496           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.042134           1    0.049546           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py

pipe_wbf_2models       mAP 0.2128 AP50 0.3995 AP_s 0.1083 (0 ms/img)


## Final choice and test submission
The chosen combination is decided ONLY by val mAP above (not by the public leaderboard).

In [7]:
pipe_res = pd.read_csv(OUT / "results.csv")
pipe_res = pipe_res[pipe_res.experiment.str.startswith("pipe_") & (pipe_res.family == "pipeline (inference)")]
print(pipe_res.sort_values("mAP", ascending=False)[["experiment", "mAP", "AP50", "AP_small", "ms_per_img", "config"]])

USE_TEMPORAL = pipe_res.set_index("experiment").mAP.get("pipe_temporal", 0) > \
    pipe_res.set_index("experiment").mAP.get(BEST_INF, 0)
print(f"final: {FINAL_RUN} + {BEST_INF}" + (f" + temporal {TEMPORAL}" if USE_TEMPORAL else ""))

def run_strategy(frames, strategy):
    if strategy == "pipe_imgsz_up":
        return predict_ultralytics(model, frames, imgsz=big)[0]
    if strategy == "pipe_tta":
        return predict_ultralytics(model, frames, imgsz=IMGSZ, augment=True)[0]
    if strategy == "pipe_tiles":
        return predict_tiled(model, frames, imgsz=IMGSZ, grid=(2, 2), overlap=0.2, merge="nms")
    return predict_ultralytics(model, frames, imgsz=IMGSZ)[0]


test_frames = list_test_frames(DATA)
test_preds = run_strategy(test_frames, BEST_INF)
if USE_TEMPORAL:
    test_preds = temporal_rescore(test_frames, test_preds, **TEMPORAL)
save_preds(test_preds, OUT / "preds_test_final.npz")
sub = write_submission(test_frames, test_preds, OUT / "submission.csv")

# sanity checks on the file format
sample = pd.read_csv(DATA / "sample_submission.csv")
assert len(sub) == len(sample) and set(sub.image_id) == set(sample.image_id)
n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) / 6
assert (n_boxes == n_boxes.round()).all() and n_boxes.max() <= 500
print(f"submission.csv: {len(sub)} rows, {n_boxes.mean():.0f} boxes per image on average")
sub.head()

          experiment     mAP    AP50  AP_small  ms_per_img  \
13     pipe_temporal  0.2348  0.4502    0.1201         0.0   
12        pipe_tiles  0.2230  0.4291    0.1161       146.5   
14  pipe_wbf_2models  0.2128  0.3995    0.1083         0.0   
10     pipe_imgsz_up  0.2115  0.3988    0.1020        36.0   
11          pipe_tta  0.2019  0.3874    0.1015        47.2   
9          pipe_base  0.2015  0.3777    0.1000        22.0   

                                               config  
13  pipe_tiles + temporal {'iou_thr': 0.2, 'weight...  
12                  full image + 2x2 tiles, NMS merge  
14                WBF(pipe_data_step3, yolo_1024_box)  
10                                    test imgsz=1280  
11                    Ultralytics TTA (flip + scales)  
9                         pipe_data_step3, imgsz=1024  
final: pipe_data_step3 + pipe_tiles + temporal {'iou_thr': 0.2, 'weight': 0.7, 'causal': False}


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/sample_submission.csv'

## Summary of all models (val, step 5 unless stated)

In [ ]:
summary = pd.read_csv(OUT / "results.csv")
summary[["experiment", "family", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "train_min"]]